# P2.3 — Modèle provisoire (baseline)
Objectif : livrer **vite** un vrai modèle utilisable, pour que P3 (FastAPI) et P1 (risk-service) puissent travailler sans attendre le champion.
Ce n'est **pas** le meilleur modèle : le benchmark (P2.4) et les tests statistiques (P2.5) désigneront le champion.

| Étape | Durée prévue |
|---|---|
| 1. Entraîner une baseline (régression logistique) | 1,5 j |
| 2. Évaluation rapide (PR-AUC, MCC) | 1 j |
| 3. Exporter le package pour P3 | 1 j |
| 4. Tester le rechargement | 0,5 j |

Prérequis : aucun (scikit-learn suffit).

> Note : une première version utilisait XGBoost 3.2.0, qui a produit un modèle constant dans cet environnement (probabilité unique, PR-AUC = 0,0017, ROC-AUC = 0,5), même sans Pipeline. La baseline est donc une régression logistique ; XGBoost sera réévalué en P2.4.

## 1. Chargement des données et du schéma

In [1]:
import sys, json, time, platform
sys.path.append("..")

import numpy as np, pandas as pd
import sklearn
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef, confusion_matrix
from sklearn.linear_model import LogisticRegression

from src.data_prep import SEED, load_clean, make_splits, build_preprocessor, input_columns

schema = json.loads(Path("../artifacts/feature_schema.json").read_text(encoding="utf-8"))
INPUT_COLS = schema["input_columns"]
assert "Time" not in INPUT_COLS, "Time doit être exclue (décision P2.2)"
assert INPUT_COLS == input_columns(False), "Le schéma ne correspond pas à data_prep.py"

df = load_clean("../datasets/creditcard.csv")
splits = make_splits(df)
X_train, y_train = splits["train"]
X_cal, y_cal = splits["cal"]
print({k: (len(v[1]), int(v[1].sum())) for k, v in splits.items()})

{'train': (170235, 284), 'cal': (56745, 94), 'test': (56746, 95)}


## 2. Entraînement de la baseline
Régression logistique (`max_iter=1000`), sans rééquilibrage (décision de P2.2). Elle a obtenu une PR-AUC d'environ 0,74 en validation croisée. On l'entraîne sur le lot d'**entraînement** uniquement.

In [2]:
pipeline = Pipeline([
    ("prep", build_preprocessor(include_time=False)),
    ("clf", LogisticRegression(max_iter=1000)),
])

t0 = time.time()
pipeline.fit(X_train, y_train)
print(f"Entraînement : {time.time() - t0:.1f} s")

Entraînement : 1.2 s


## 3. Évaluation rapide
On évalue sur le lot de **calibration** (non utilisé pour entraîner, ni pour décider). Le lot de **test** reste intact pour l'évaluation finale du champion.

Repère : un classifieur au hasard aurait une PR-AUC d'environ 0,0017 (le taux de fraude).

In [3]:
proba_cal = pipeline.predict_proba(X_cal)[:, 1]
pred_cal = (proba_cal >= 0.5).astype(int)

metrics = {
    "pr_auc": round(float(average_precision_score(y_cal, proba_cal)), 4),
    "roc_auc": round(float(roc_auc_score(y_cal, proba_cal)), 4),
    "mcc_seuil_0.5": round(float(matthews_corrcoef(y_cal, pred_cal)), 4),
}
print(metrics)

# Garde-fous : un modèle dégénéré (probabilités constantes) doit arrêter le notebook
assert np.unique(proba_cal).size > 100, "Probabilités quasi constantes : modèle dégénéré"
assert metrics["roc_auc"] > 0.9, "ROC-AUC trop basse : modèle dégénéré ?"

tn, fp, fn, tp = confusion_matrix(y_cal, pred_cal).ravel()
pd.DataFrame({"prédit légitime": [tn, fn], "prédit fraude": [fp, tp]}, index=["réel légitime", "réel fraude"])

{'pr_auc': 0.7978, 'roc_auc': 0.9747, 'mcc_seuil_0.5': 0.7723}


,prédit légitime,prédit fraude
réel légitime,56642,9
réel fraude,30,64


### Latence d'inférence (une transaction à la fois)
Important pour P3 : l'appel à `/predict` a un timeout de **800 ms** côté risk-service.

In [4]:
rows = [X_cal.iloc[[i]] for i in range(200)]
times = []
for r in rows:
    t0 = time.perf_counter()
    pipeline.predict_proba(r)
    times.append((time.perf_counter() - t0) * 1000)

latency = {"median_ms": round(float(np.median(times)), 2), "p95_ms": round(float(np.percentile(times, 95)), 2)}
print(latency)

{'median_ms': 10.93, 'p95_ms': 15.82}


## 4. Incertitude provisoire (placeholder)
Le contrat d'API exige `uncertainty_score` et `is_uncertain`, mais le conformal (MAPIE) n'existe qu'en P2.6.
En attendant, on propose une règle **simple et documentée**, à valider avec P1 et P3 :

- `uncertainty_score = 1 - |2p - 1|` : 0 quand le modèle est sûr (p proche de 0 ou 1), 1 quand p = 0,5 ;
- `is_uncertain = uncertainty_score >= 0.6` (soit p entre 0,3 et 0,7).

Elle sera **remplacée** par le conformal dans le modèle champion, sans changer le format de sortie.

In [5]:
UNCERTAINTY_THRESHOLD = 0.6

def placeholder_uncertainty(p):
    score = 1 - np.abs(2 * np.asarray(p) - 1)
    return score, score >= UNCERTAINTY_THRESHOLD

score, flag = placeholder_uncertainty(proba_cal)
print(f"Part des transactions marquées incertaines (calibration) : {flag.mean():.3%}")
print(f"Fraudes parmi les incertaines : {int(y_cal[flag].sum())} / {int(flag.sum())}")

pd.DataFrame({"proba": proba_cal[:8].round(4), "uncertainty_score": score[:8].round(3), "is_uncertain": flag[:8]})

Part des transactions marquées incertaines (calibration) : 0.028%
Fraudes parmi les incertaines : 6 / 16


,proba,uncertainty_score,is_uncertain
0,0.0001,0.0,False
1,0.0000,0.0,False
2,0.0001,0.0,False
3,0.0001,0.0,False
4,0.0002,0.0,False
5,0.0000,0.0,False
6,0.0000,0.0,False
7,0.0001,0.0,False


In [8]:
def decide(p, is_uncertain):
    if is_uncertain: return "HUMAN_REVIEW"
    if p >= 0.90: return "BLOCKED"
    if p <= 0.50: return "APPROVED"
    return "HUMAN_REVIEW"

decisions = pd.Series([decide(p, u) for p, u in zip(proba_cal, flag)])
print(pd.crosstab(decisions, y_cal.to_numpy(), rownames=["décision"], colnames=["réel (0=légitime, 1=fraude)"]))
print((decisions.value_counts(normalize=True) * 100).round(3))

réel (0=légitime, 1=fraude)      0   1
décision                              
APPROVED                     56635  28
BLOCKED                          5  51
HUMAN_REVIEW                    11  15
APPROVED        99.855
BLOCKED          0.099
HUMAN_REVIEW     0.046
Name: proportion, dtype: float64


## 5. Export du package pour P3

In [6]:
import joblib, shutil

PKG = Path("../artifacts/provisional")
PKG.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, PKG / "model.joblib")
shutil.copy("../artifacts/feature_schema.json", PKG / "feature_schema.json")

meta = {
    "model_version": "provisional-0.1.0",
    "status": "provisional - sera remplace par le champion (P2.7)",
    "model_type": "LogisticRegression (max_iter=1000) dans un sklearn Pipeline",
    "trained_on": f"lot d'entrainement, seed {SEED}",
    "input_columns": INPUT_COLS,
    "how_to_predict": "pipeline.predict_proba(DataFrame avec input_columns)[:, 1] = probabilite de fraude",
    "metrics_on_calibration_set": metrics,
    "latency_single_row": latency,
    "uncertainty": {
        "type": "placeholder",
        "score": "1 - abs(2p - 1)",
        "is_uncertain_if": f"score >= {UNCERTAINTY_THRESHOLD}",
        "note": "remplace par le conformal (MAPIE) dans le modele champion",
    },
    "versions": {
        "python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}
(PKG / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

archive = shutil.make_archive("../artifacts/provisional_package", "zip", PKG)
print("Package :", archive)
print([p.name for p in PKG.iterdir()])

Package : c:\Users\hp\Documents\fintech-mlops-platform\ml-research\artifacts\provisional_package.zip
['feature_schema.json', 'meta.json', 'model.joblib']


## 6. Test de rechargement
Simule ce que fera P3 : recharger le modèle depuis le fichier et vérifier qu'il donne les mêmes probabilités.

In [7]:
reloaded = joblib.load(PKG / "model.joblib")
sample = X_cal[INPUT_COLS].head(5)          # colonnes BRUTES, comme l'enverra l'API

p_orig = pipeline.predict_proba(sample)[:, 1]
p_reload = reloaded.predict_proba(sample)[:, 1]
assert np.allclose(p_orig, p_reload), "Le modèle rechargé donne des résultats différents"
print("OK : modèle rechargé identique")
print(p_reload.round(4))

OK : modèle rechargé identique
[0.0001 0.     0.0001 0.0001 0.0002]


## 7. Conclusions de P2.3
- **Modèle** : régression logistique (max_iter=1000) dans un Pipeline (prétraitement + classifieur), entraînée sur le lot d'entraînement, sans rééquilibrage. XGBoost 3.2.0 a produit un modèle constant dans notre environnement (probabilité unique, PR-AUC = 0,0017) : à réévaluer en P2.4.
- **Performance (lot de calibration, 94 fraudes)** : PR-AUC = 0,7978 (hasard ≈ 0,0017), ROC-AUC = 0,9747, MCC à 0,5 = 0,7723.
- **Matrice de confusion** : 64 fraudes détectées sur 94 (rappel 68 %), 30 ratées, 9 fausses alertes sur 56 651 transactions légitimes (précision 88 %).
- **Latence** : médiane 10,9 ms, p95 15,8 ms par transaction (timeout du risk-service : 800 ms).
- **Incertitude** : règle provisoire 1 - |2p - 1|, seuil 0,6 (p entre 0,3 et 0,7) ; seulement 0,028 % des transactions sont marquées incertaines (16, dont 6 fraudes), très en deçà de la cible de ~1,7 %, qui sera atteinte avec le conformal (P2.6).
- **Livrable** : `ml-research/artifacts/provisional_package.zip` (model.joblib, feature_schema.json, meta.json).
- **Limites** : modèle non optimisé, seuil 0,5 non réglé, 94 fraudes dans le lot d'évaluation (résultats variables), pas de conformal, pas de comparaison avec d'autres algorithmes (P2.4).
- **Décision complète (règle du document : 0,5 / 0,9 + incertitude provisoire)** sur le lot de calibration : 99,855 % approuvées, 0,099 % bloquées (51 fraudes et 5 légitimes), 0,046 % en revue humaine (15 fraudes et 11 légitimes) ; 28 fraudes sur 94 sont approuvées à tort, car leur probabilité est trop basse pour être signalée. La revue humaine reste très en deçà de la cible de ~1,7 %.